# 02b · RetinaNet (one-stage, anchor-based)

Рубрика: *One-stage Anchor-based*. Протокол такий самий, як для Faster R-CNN (ноутбук 02): 3-fold GroupKFold за кліпами
(`splits.json`), тренувальні кадри — кожен 20-й, валідаційні — кожен 10-й, 640 px, 12 епох, OOF COCO-метрики з ignored
regions як `iscrowd=1` (`common.evaluate`).

Код тренування й оцінювання — у `tv_train.py` (спільний із Faster R-CNN), завантаження даних — у `tv_dataset.py`,
метрики й логування — у `common.py`.

Як запускати: задати `MODE`, `EXPERIMENT` і `FOLDS` нижче. `quick` за кілька хвилин перевіряє всі гілки коду;
`full` тренує один експеримент на всіх фолдах (один експеримент на один Kaggle commit).

## 1. Конфігурація запуску
`MODE` — `"quick"` або `"full"`; `EXPERIMENT` — ключ зі словника `EXPERIMENTS` (розділ 3) або список ключів, які
тренуються один за одним в одному commit (у QUICK — лише перший); `FOLDS` — фолди, які тренуються в цій сесії. `RUN_ANALYSIS` вмикає аналіз анкорів, помилок і порівняння з Faster R-CNN (у quick-режимі завжди увімкнено).
`BEST_ANCHORS` — найкращий варіант анкорів після експериментів 1–2, на ньому будуються експерименти 3–4.
`FRCNN_EXPERIMENT` — модель Faster R-CNN для порівняння (`auto` = найкраща за OOF mAP у `results.csv`).

In [ ]:
MODE = "quick"
EXPERIMENT = ["retina_base", "retina_anch05", "retina_g1"]
FOLDS = [0, 1, 2]
RUN_ANALYSIS = True
ANALYSIS_EXPERIMENT = "auto"
BEST_ANCHORS = "retina_anch05"
FRCNN_EXPERIMENT = "auto"

## 2. Підготовка
Беремо спільні модулі поруч із ноутбуком (репозиторій) або завантажуємо їх із GitHub (Kaggle).

In [ ]:
import os
import sys
import time
import urllib.request
from pathlib import Path

REPO_RAW = "https://raw.githubusercontent.com/KaplyshOlha/hw-2/main/notebooks/"
SRC_FILES = ["common.py", "tv_dataset.py", "tv_train.py", "splits.json", "anchors.json"]
SRC = next((Path(p) for p in [".", "notebooks"] if all((Path(p) / f).exists() for f in SRC_FILES)), None)
if SRC is None:
    SRC = Path("/tmp/hw2_src")
    SRC.mkdir(parents=True, exist_ok=True)
    for f in SRC_FILES:
        urllib.request.urlretrieve(REPO_RAW + f, SRC / f)
sys.path.insert(0, str(SRC.resolve()))

Імпорти, шляхи для результатів і пристрій. У FULL-режимі ноутбук одразу зупиняється, якщо GPU не підключено.

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

import common
import tv_train as tvt

assert MODE in ("quick", "full")
assert hasattr(tvt, "anchor_match_stats"), "old tv_train.py: push the new version to GitHub first"
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
RUN_ROOT = WORK / ("runs" if MODE == "full" else "runs_quick")
RESULTS = WORK / ("results.csv" if MODE == "full" else "results_quick.csv")
IMG_DIR = WORK / "images"
IMG_DIR.mkdir(parents=True, exist_ok=True)
INPUT_ROOT = os.environ.get("HW2_INPUTS", "/kaggle/input")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if MODE == "full":
    assert DEVICE.type == "cuda", "FULL mode needs a GPU (Settings -> Accelerator -> GPU T4)"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 130, "savefig.bbox": "tight"})
print(f"MODE={MODE} EXPERIMENT={EXPERIMENT} FOLDS={FOLDS} "
      f"device={torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'cpu'} src={SRC}")

Копіюємо завершені або перервані фолди (і `results.csv`) з попередніх версій ноутбуків, підключених як input
(і RetinaNet, і Faster R-CNN), щоб нова сесія продовжувала роботу, а не тренувала заново.

In [ ]:
tvt.restore_from_inputs(RUN_ROOT, RESULTS, INPUT_ROOT, name=RUN_ROOT.name);

## 3. Експерименти
Протокол оптимізації такий самий, як у Faster R-CNN (`tvt.Config`): SGD, momentum 0.9, wd 1e-4, batch 8, LR 0.01,
200 warmup-ітерацій, 12 епох із ×0.1 на епохах 8 і 11, AMP, горизонтальний flip, ignored regions зафарбовані сірим
у тренувальних кадрах, валідаційні кадри «сирі», як тестові. `score_thresh=0.001`, `detections_per_img=300`, NMS 0.5.

Модель — `retinanet_resnet50_fpn` з COCO-вагами: backbone, FPN (P3–P7), regression head і чотири conv-шари
classification head беруться з COCO; новий лише останній conv класифікації (8 класів, sigmoid), його bias
ініціалізовано prior π = 0.01. Кожен експеримент змінює одну річ відносно попереднього кроку.

| експеримент | зміна | мотивація |
|---|---|---|
| `retina_base` | дефолтні анкори torchvision: 32–512 px (3 масштаби × 3 ratio на рівень) | референс |
| `retina_anch05` | анкори ×0.5 → 16–406 px | EDA: медіана об'єкта 15 px при 640; дефолтні анкори покривають 0% малих об'єктів, ×0.5 — 65% усіх |
| `retina_kmeans` | анкори з k-means (k=9) по боксах тренувальної частини фолду → 4–55 px | YOLOv2: анкори з розподілу даних; фітимо лише на train-частині кожного фолду. Ризик: розміри розкладено по рівнях за квантилями, тож анкори 15–55 px опиняються на P5–P7 (stride 32–128), де вони рідкі |
| `retina_iou43` | `BEST_ANCHORS` + IoU-пороги fg/bg 0.5/0.4 → 0.4/0.3 | у малих об'єктів менше анкорів з IoU ≥ 0.5 (розділ 5) |
| `retina_g1` | `BEST_ANCHORS` + focal γ 2 → 1 | обов'язковий loss-експеримент: на кадр сотні позитивів, дисбаланс менший, ніж у COCO |
| `retina_a05` *(опц.)* | `BEST_ANCHORS` + focal α 0.25 → 0.5 | більша вага позитивів → recall малих об'єктів |
| `retina_giou` *(опц.)* | `BEST_ANCHORS` + GIoU замість L1 на дельтах | те саме порівняння, що `frcnn_giou` у ноутбуці 02 |
| `retina_anch025` *(опц.)* | анкори ×0.25 → 8–203 px | проміжний варіант між ×0.5 і k-means |
| `retina_tuned` *(опц.)* | `BEST_ANCHORS` + зміни, що допомогли (`TUNED_CHANGES`) | фінальна one-stage модель |

Аспектні співвідношення не тюнимо: k-means дає ≈ (0.5, 1, 2), тобто те саме, що дефолт; тюнимо розмір анкорів.

In [ ]:
BASE = tvt.Config(experiment="retina_base", model="retinanet", anchors_json=str(SRC / "anchors.json"),
                  notes="torchvision RetinaNet defaults, COCO-pretrained, 640px")
EXPERIMENTS = {"retina_base": BASE}
EXPERIMENTS["retina_anch05"] = tvt.make_config(
    BASE, experiment="retina_anch05", anchor_key="x0.5", notes="anchors x0.5 (16-406)")
EXPERIMENTS["retina_kmeans"] = tvt.make_config(
    BASE, experiment="retina_kmeans", anchor_key="kmeans", notes="per-fold k-means anchors (4-55)")
EXPERIMENTS["retina_anch025"] = tvt.make_config(
    BASE, experiment="retina_anch025", anchor_key="x0.25", notes="anchors x0.25 (8-203)")
BEST = EXPERIMENTS[BEST_ANCHORS]
EXPERIMENTS["retina_iou43"] = tvt.make_config(
    BEST, experiment="retina_iou43", fg_iou_thresh=0.4, bg_iou_thresh=0.3,
    notes=f"{BEST_ANCHORS} + fg/bg IoU 0.4/0.3")
EXPERIMENTS["retina_g1"] = tvt.make_config(
    BEST, experiment="retina_g1", focal_gamma=1.0, notes=f"{BEST_ANCHORS} + focal gamma 1")
EXPERIMENTS["retina_a05"] = tvt.make_config(
    BEST, experiment="retina_a05", focal_alpha=0.5, notes=f"{BEST_ANCHORS} + focal alpha 0.5")
EXPERIMENTS["retina_giou"] = tvt.make_config(
    BEST, experiment="retina_giou", retina_reg_loss="giou", notes=f"{BEST_ANCHORS} + GIoU box loss")
TUNED_CHANGES = dict(fg_iou_thresh=0.4, bg_iou_thresh=0.3, focal_gamma=1.0)
EXPERIMENTS["retina_tuned"] = tvt.make_config(
    BEST, experiment="retina_tuned", **TUNED_CHANGES, notes=f"{BEST_ANCHORS} + {TUNED_CHANGES}")

QUICK = dict(epochs=2, max_iters=15, batch_size=2, train_stride=100, eval_stride=200,
             warmup_iters=10, milestones=(1,), fps_warmup=3, fps_iters=10)


def get_cfg(name):
    c = EXPERIMENTS[name]
    return tvt.make_config(c, **QUICK) if MODE == "quick" else c


RUN_LIST = [EXPERIMENT] if isinstance(EXPERIMENT, str) else list(EXPERIMENT)
RUN_LIST = RUN_LIST[:1] if MODE == "quick" else RUN_LIST
assert all(n in EXPERIMENTS for n in RUN_LIST), f"unknown experiment in {RUN_LIST}"
cfg = get_cfg(RUN_LIST[0])
pd.Series({k: v for k, v in cfg.key().items() if v != tvt.Config().key().get(k)}, name="changed vs defaults", dtype=object)

## 4. Дані та фолди
Завантажуємо всі розмічені кадри (train + val) і фіксований 3-fold split за кліпами; показуємо, скільки кадрів кожен
фолд тренує й валідує з кроками цього запуску.

In [ ]:
common.seed_everything()
images, anns = common.load_gt(cache="/tmp/gt_cache.pkl")
splits = common.load_splits(SRC / "splits.json")
pd.DataFrame([{"fold": f["fold"], "train_clips": len(f["train_clips"]), "val_clips": len(f["val_clips"]),
               "train_frames": len(common.fold_images(images, splits, f["fold"], "train", cfg.train_stride)),
               "val_frames": len(common.fold_images(images, splits, f["fold"], "val", cfg.eval_stride))}
              for f in splits["folds"]]).set_index("fold")

## 5. Анкори й розмір об'єктів (без тренування)
RetinaNet призначає позитиви лише через IoU з анкорами: анкор позитивний, якщо IoU ≥ fg (0.5), негативний, якщо
IoU < bg (0.4), а проміжні ігноруються. Якщо в GT немає жодного анкора з IoU ≥ fg, torchvision усе одно робить
позитивним його найкращий анкор (*low-quality match*), але це один слабкий приклад замість кількох.

Тут ми застосовуємо той самий `Matcher` до GT-боксів тренувальної частини фолду 0 (640 px, ignored regions уже
відфільтровані) для кожного варіанту анкорів і двох пар порогів. Модель не потрібна, тому це обґрунтування вибору
анкорів ще до тренування; k-means анкори пораховані лише на train-частині фолду.

In [ ]:
ANCHOR_VARIANTS = {"default": "retina_base", "x0.5": "retina_anch05", "x0.25": "retina_anch025", "kmeans": "retina_kmeans"}
DO_ANALYSIS = RUN_ANALYSIS or MODE == "quick"
if DO_ANALYSIS:
    t0 = time.time()
    atr, _ = tvt.make_datasets(get_cfg("retina_base"), images, anns, splits, 0, precache=False)
    agt, aimg = tvt.anchor_match_stats({k: get_cfg(v) for k, v in ANCHOR_VARIANTS.items()}, atr, fold=0,
                                       thresholds=((0.5, 0.4), (0.4, 0.3)),
                                       max_images=300 if MODE == "full" else 30)
    g05 = agt[agt["fg"] == 0.5]
    natural = g05.pivot_table(index="size_bin", columns="variant", values="natural_pos", aggfunc="mean", observed=True)
    natural.index = natural.index.astype(str)
    natural.loc["all"] = g05.groupby("variant")["natural_pos"].mean()
    pos = agt.pivot_table(index="size_bin", columns=["variant", "fg"], values="n_pos", aggfunc="mean", observed=True)
    pos.index = pos.index.astype(str)
    pos.loc["all"] = agt.groupby(["variant", "fg"])["n_pos"].mean()
    imb = aimg.groupby(["variant", "fg"])[["anchors", "pos", "between", "neg"]].mean()
    imb["neg_per_pos"] = imb["neg"] / imb["pos"].clip(lower=1)
    n_gt = g05[g05["variant"] == "default"].groupby("size_bin", observed=True).size()
    natural.insert(0, "n_gt", n_gt.rename(index=str).reindex(natural.index))
    natural.loc["all", "n_gt"] = n_gt.sum()
    print(f"{int(n_gt.sum())} GT boxes on {(aimg['variant'] == 'default').sum() // 2} frames, {time.time() - t0:.0f}s")
    print("Share of GT boxes with at least one anchor at IoU >= 0.5 (by GT size at 640 px):")
    display(natural[["n_gt"] + list(ANCHOR_VARIANTS)].round(3))
    print("Mean positive anchors per GT box (fg = 0.5 / 0.4):")
    display(pos[[c for c in pos.columns if c[0] in ANCHOR_VARIANTS]].reindex(columns=list(ANCHOR_VARIANTS), level=0).round(2))
    print("Anchors per frame and foreground/background imbalance:")
    display(imb.reindex(list(ANCHOR_VARIANTS), level=0).round(1))

Частка GT з «природним» позитивом і середня кількість позитивних анкорів на GT за розміром об'єкта
(суцільні лінії — fg/bg 0.5/0.4, пунктир — 0.4/0.3).

In [ ]:
if DO_ANALYSIS:
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
    for i, v in enumerate(ANCHOR_VARIANTS):
        c = f"C{i}"
        ax[0].plot(natural.index[:-1].astype(str), natural[v].iloc[:-1], marker="o", color=c, label=v)
        for fg, ls in [(0.5, "-"), (0.4, "--")]:
            if (v, fg) in pos.columns:
                ax[1].plot(pos.index[:-1].astype(str), pos[(v, fg)].iloc[:-1], marker="o", ls=ls, color=c,
                           label=f"{v} fg={fg}")
    ax[0].set_title("GT with an anchor at IoU ≥ 0.5")
    ax[1].set_title("positive anchors per GT")
    for a in ax:
        a.set_xlabel("GT size sqrt(w·h) at 640 px")
    ax[0].set_ylim(0, 1.02)
    ax[0].legend(fontsize=8)
    ax[1].legend(fontsize=7, ncol=2)
    fig.savefig(IMG_DIR / "02b_retina_anchor_matching.png")
    plt.show()

**Спостереження (анкори):** TBD

## 6. Тренування й передбачення
Для кожного експерименту з `RUN_LIST` тренуємо кожен фолд із `FOLDS` (завершені пропускаються, перервані продовжуються з `last.pt`), передбачаємо
валідаційні кадри фолду й міряємо FPS. У QUICK-режимі спершу імітується переривання після епохи 1, щоб перевірити й resume.

In [ ]:
t0 = time.time()
if MODE == "quick":
    try:
        tvt.run_experiment(tvt.make_config(cfg, stop_after_epoch=1), images, anns, splits, FOLDS[:1],
                           RUN_ROOT, RESULTS, DEVICE)
    except tvt.Interrupted as e:
        print("resume test:", e)
all_metrics = {}
for name in RUN_LIST:
    all_metrics[name] = tvt.run_experiment(get_cfg(name), images, anns, splits, FOLDS, RUN_ROOT, RESULTS, DEVICE,
                                           require_all_folds=(MODE == "full"))
    print(f"{name} finished, {(time.time() - t0) / 60:.1f} min since start")

OOF-метрики експериментів цього запуску: по фолдах, mean ± std по фолдах і pooled out-of-fold (`oof`).

In [ ]:
MAIN = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR500"]
PER_CLASS = [f"AP_{c}" for c in common.CLASSES]
for name, metrics in all_metrics.items():
    if metrics is not None:
        print(name)
        display(metrics[MAIN].round(4))
        display(metrics.loc[["oof"], PER_CLASS].round(4))

## 7. Порівняння експериментів RetinaNet
Останній залогований запуск кожного експерименту RetinaNet з `results.csv` (OOF і mean ± std по фолдах), per-class AP.

In [ ]:
names = [n for n in EXPERIMENTS if (RUN_ROOT / n).exists()]
summ = tvt.summary(RESULTS, model="retinanet", experiments=names)
num = summ.select_dtypes("number").columns
summ[num] = summ[num].where(summ[num] >= 0)
if len(summ):
    table = pd.DataFrame({m: summ[f"{m}_oof"].round(4) for m in ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR500"]})
    table["mAP mean±std"] = [f"{a:.4f} ± {b:.4f}" for a, b in zip(summ["mAP_mean"], summ["mAP_std"])]
    table["FPS"] = summ["fps"].round(1)
    table["train min/fold"] = summ["train_min"].round(1)
    table.to_csv(WORK / "02b_retina_comparison.csv")
    display(table)
    display(summ[[c for c in PER_CLASS if c in summ]].round(4))

Основні метрики (std по фолдах як error bars) і OOF AP по класах для кожного експерименту.

In [ ]:
if len(summ):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [1, 1.3]})
    ms = ["mAP", "AP50", "AP_small", "AP_medium", "AP_large"]
    x = np.arange(len(ms))
    w = 0.8 / len(summ)
    for i, (n, r) in enumerate(summ.iterrows()):
        ax[0].bar(x + i * w, [r[f"{m}_mean"] for m in ms], w, yerr=[r[f"{m}_std"] for m in ms],
                  capsize=3, label=n)
    ax[0].set_xticks(x + w * (len(summ) - 1) / 2, ms)
    ax[0].set_title("RetinaNet: mean over folds (± std)")
    ax[0].legend(fontsize=8)
    pc = summ[[c for c in PER_CLASS if c in summ]].astype(float)
    im = ax[1].imshow(pc.values, cmap="viridis", aspect="auto")
    ax[1].set_xticks(range(pc.shape[1]), [c[3:] for c in pc.columns], rotation=30)
    ax[1].set_yticks(range(len(pc)), pc.index)
    for (i, j), v in np.ndenumerate(pc.values):
        ax[1].text(j, i, f"{v:.2f}", ha="center", va="center", color="w", fontsize=8)
    ax[1].set_title("OOF AP per class")
    fig.colorbar(im, ax=ax[1])
    fig.savefig(IMG_DIR / "02b_retina_comparison.png")
    plt.show()

Тренувальний loss по епохах (фолд 0) для кожного завершеного експерименту: сумарний, focal (класифікація) і регресія.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
for n in names:
    done = tvt.load_done(RUN_ROOT, n)
    if 0 not in done:
        continue
    h = pd.DataFrame(done[0]["history"])
    for a, col in zip(ax, ["loss", "classification", "bbox_regression"]):
        if col in h:
            a.plot(h["epoch"], h[col], marker="o", ms=3, label=n)
            a.set_title(col)
            a.set_xlabel("epoch")
ax[0].legend(fontsize=8)
fig.savefig(IMG_DIR / "02b_retina_loss_curves.png")
plt.show()

Примітка: loss різних експериментів не завжди порівнюваний напряму — зміна γ/α змінює масштаб focal loss,
а GIoU — масштаб регресійного loss. Порівнюємо за mAP, а криві — лише щоб побачити збіжність.

**Спостереження (порівняння RetinaNet):** TBD

## 8. RetinaNet проти Faster R-CNN
Беремо результати обох моделей з об'єднаного `results.csv` (локальний + копія з репозиторію): початковий і найкращий
RetinaNet проти базового і найкращого Faster R-CNN (inference-only записи `*_propN` не враховуються), з розбивкою за
розміром об'єктів (COCO: small < 32², medium < 96², large — решта, в оригінальних пікселях) і по класах.

In [ ]:
ALL_RESULTS = WORK / "results_all.csv"
ALL_RESULTS.unlink(missing_ok=True)
try:
    urllib.request.urlretrieve(REPO_RAW + "results.csv", ALL_RESULTS)
except Exception as e:
    print("repo results.csv not available:", e)
if RESULTS.exists():
    common.merge_results(str(RESULTS), str(ALL_RESULTS))
rt = tvt.summary(ALL_RESULTS, model="retinanet") if ALL_RESULTS.exists() else pd.DataFrame()
fr = tvt.summary(ALL_RESULTS, model="faster_rcnn") if ALL_RESULTS.exists() else pd.DataFrame()
if len(fr):
    fr = fr[~fr.index.str.contains("_prop")]
for t in (rt, fr):
    num = t.select_dtypes("number").columns
    t[num] = t[num].where(t[num] >= 0)
RETINA_BEST = rt["mAP_oof"].idxmax() if len(rt) else None
FRCNN_BEST = (fr["mAP_oof"].idxmax() if FRCNN_EXPERIMENT == "auto" else FRCNN_EXPERIMENT) if len(fr) else None
picks = {}
if "retina_base" in rt.index:
    picks["RetinaNet initial (retina_base)"] = rt.loc["retina_base"]
if RETINA_BEST is not None and RETINA_BEST != "retina_base":
    picks[f"RetinaNet tuned ({RETINA_BEST})"] = rt.loc[RETINA_BEST]
if "frcnn_base" in fr.index and FRCNN_BEST != "frcnn_base":
    picks["Faster R-CNN initial (frcnn_base)"] = fr.loc["frcnn_base"]
if FRCNN_BEST is not None:
    picks[f"Faster R-CNN tuned ({FRCNN_BEST})"] = fr.loc[FRCNN_BEST]
cmp = pd.DataFrame(picks).T
if len(cmp):
    SIZE_M = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR500"]
    cmp_table = pd.DataFrame({m: cmp[f"{m}_oof"].astype(float).round(4) for m in SIZE_M})
    cmp_table["mAP mean±std"] = [f"{a:.4f} ± {b:.4f}" for a, b in zip(cmp["mAP_mean"], cmp["mAP_std"])]
    cmp_table["FPS"] = cmp["fps"].astype(float).round(1)
    cmp_table["train min/fold"] = cmp["train_min"].astype(float).round(1)
    cmp_table.to_csv(WORK / "02b_retina_vs_frcnn.csv")
    display(cmp_table)
    display(cmp[[c for c in PER_CLASS if c in cmp]].astype(float).round(4))
else:
    print("no logged runs to compare yet")

Метрики за розміром (std по фолдах) і різниця AP по класах: найкращий RetinaNet мінус найкращий Faster R-CNN.

In [ ]:
if len(cmp):
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1.3, 1]})
    ms = ["mAP", "AP50", "AP_small", "AP_medium", "AP_large"]
    x = np.arange(len(ms))
    w = 0.8 / len(cmp)
    for i, (n, r) in enumerate(cmp.iterrows()):
        ax[0].bar(x + i * w, [float(r[f"{m}_mean"]) for m in ms], w,
                  yerr=[float(r[f"{m}_std"]) for m in ms], capsize=3, label=n)
    ax[0].set_xticks(x + w * (len(cmp) - 1) / 2, ms)
    ax[0].set_title("mean over folds (± std)")
    ax[0].legend(fontsize=7)
    if RETINA_BEST is not None and FRCNN_BEST is not None:
        d = (rt.loc[RETINA_BEST, PER_CLASS].astype(float) - fr.loc[FRCNN_BEST, PER_CLASS].astype(float))
        ax[1].barh([c[3:] for c in PER_CLASS], d.values, color=np.where(d.values >= 0, "tab:green", "tab:red"))
        ax[1].axvline(0, color="k", lw=0.8)
        ax[1].set_title(f"AP per class: {RETINA_BEST} − {FRCNN_BEST}")
    fig.savefig(IMG_DIR / "02b_retina_vs_frcnn.png")
    plt.show()

**Спостереження (RetinaNet vs Faster R-CNN):** TBD

### 8.1 Що змінюється без proposal stage
**Дисбаланс foreground/background.** У Faster R-CNN RPN відсікає майже весь фон, і другий етап навчається на
512 RoI на кадр із контрольованою часткою позитивів (25%). RetinaNet класифікує *кожен* анкор: при 640 px це десятки
тисяч анкорів на кадр (таблиця в розділі 5), з яких позитивні лише сотні. Зі звичайним cross-entropy сумарний
loss і градієнт складалися б здебільшого з легких негативів (небо, дорога), і модель навчилася б просто казати «фон».

**Немає другого уточнення.** У two-stage детектора бокс уточнюється двічі (RPN, потім RoI head), а ознаки для
класифікації беруться RoIAlign саме з області пропозалу на відповідному рівні FPN. У RetinaNet клас і бокс
передбачаються одним проходом з ознак у точці анкора; для об'єкта 10–15 px це кілька пікселів P3 (stride 8), тож
помилка анкора не виправляється другим етапом.

**Дублікати й щільні сцени.** Сусідні анкори дають багато перекритих боксів для одного об'єкта, і прибирає їх лише
NMS (IoU 0.5). У щільних сценах NMS або залишає дублікати, або гасить справжніх сусідів (парковки, натовпи).

**Як це вирішує RetinaNet і що ще можна зробити.** (1) Focal loss множить CE на (1 − p)^γ, тож легкі негативи майже
не дають градієнта, а α балансує позитиви й негативи; loss нормується на кількість позитивів, а не анкорів.
(2) Prior-ініціалізація bias класифікатора (π = 0.01) — на старті всі анкори мають низький score, і перші ітерації
не «вибухають» від мільйонів негативів. (3) Анкори між порогами bg і fg ігноруються. (4) Анкори під розмір об'єктів
(експерименти 1–2) і нижчі IoU-пороги (експеримент 3) дають малим об'єктам більше позитивів. Далі можливі адаптивне
призначення (ATSS), оцінка якості локалізації (GFL), каскадне уточнення (RefineDet), Soft-NMS — частина з них
перевіряється в наступних ноутбуках.

## 9. Аналіз помилок
Беремо найкращий RetinaNet (`auto`) і, якщо його передбачення є в `runs/` (підключено output ноутбука 02), найкращий
Faster R-CNN. Аналізу потрібні збережені `preds.csv.gz` фолдів.

In [ ]:
if ANALYSIS_EXPERIMENT == "auto":
    ANALYSIS_EXPERIMENT = summ["mAP_oof"].idxmax() if len(summ) else RUN_LIST[0]
acfg = get_cfg(ANALYSIS_EXPERIMENT)
apreds = tvt.load_fold_preds(RUN_ROOT, ANALYSIS_EXPERIMENT)
fpreds = tvt.load_fold_preds(RUN_ROOT, FRCNN_BEST) if FRCNN_BEST else {}
DO_ANALYSIS = DO_ANALYSIS and len(apreds) > 0
HAVE_FRCNN = DO_ANALYSIS and sorted(fpreds) == sorted(apreds)
print(f"analysis of {ANALYSIS_EXPERIMENT}, folds {sorted(apreds)}, enabled={DO_ANALYSIS}; "
      f"Faster R-CNN {FRCNN_BEST}: folds {sorted(fpreds)}, compared={HAVE_FRCNN}")

### 9.1 Поріг score
Sigmoid-score focal loss і softmax-score Faster R-CNN мають різну калібровку (у RetinaNet правильні детекції часто
мають 0.3–0.5), тож фіксований поріг був би нечесним. Для кожної моделі обираємо поріг із найкращим F1
(IoU ≥ 0.5, той самий клас) на кожному 4-му OOF-кадрі.

In [ ]:
if DO_ANALYSIS:
    oof_ids = np.concatenate([common.fold_images(images, splits, k, "val", acfg.eval_stride)["img_id"].to_numpy()
                              for k in sorted(apreds)])
    models = {ANALYSIS_EXPERIMENT: pd.concat(apreds.values(), ignore_index=True)}
    if HAVE_FRCNN:
        models[FRCNN_BEST] = pd.concat(fpreds.values(), ignore_index=True)
    thr, thr_tabs = {}, {}
    for n, p in models.items():
        thr[n], thr_tabs[n] = tvt.f1_threshold(p, images, anns, oof_ids)
    display(pd.concat(thr_tabs, axis=1).round(3))
    print("F1-optimal thresholds:", thr)

### 9.2 Типи помилок
Детекції з score ≥ порогу жадібно матчаться з GT (IoU ≥ 0.5, той самий клас). FP типізуються як `cls` (перекриває GT
іншого класу), `loc` (той самий клас, 0.1 ≤ IoU < 0.5), `dup` (GT уже знайдено), `bkg`; детекції переважно всередині
ignored regions відкидаються. Розміри — в оригінальних пікселях.

In [ ]:
if DO_ANALYSIS:
    errs, rows = {}, {}
    for n, p in models.items():
        gt, dt = tvt.error_analysis(p, images, anns, oof_ids, score_thr=thr[n])
        gt["density"] = pd.cut(gt["n_in_frame"], [0, 20, 50, 100, 1000], labels=["≤20", "21-50", "51-100", ">100"])
        errs[n] = (gt, dt)
        rows[n] = {"thr": thr[n], "recall": gt["matched"].mean(),
                   "precision": (dt["type"] == "tp").mean() if len(dt) else np.nan,
                   "dets/frame": len(dt) / len(oof_ids),
                   **dt["type"].value_counts(normalize=True).rename(lambda s: f"dets {s}").to_dict()}
    gt, dt = errs[ANALYSIS_EXPERIMENT]
    display(pd.DataFrame(rows).T.round(3))

Recall за розміром, оклюзією і щільністю сцени для обох моделей та матриця сплутування класів для RetinaNet.

In [ ]:
if DO_ANALYSIS:
    fig, ax = plt.subplots(1, 4, figsize=(19, 3.8), gridspec_kw={"width_ratios": [1, 0.6, 0.8, 1.2]})
    for a, col, title in zip(ax[:3], ["size_bin", "occlusion", "density"],
                             ["GT size (orig px)", "occlusion (0 none, 1 part, 2 heavy)", "objects per frame"]):
        w = 0.8 / len(errs)
        for i, (n, (g, _)) in enumerate(errs.items()):
            r = g.groupby(col, observed=True)["matched"].agg(["mean", "size"])
            xs = np.arange(len(r)) + i * w
            a.bar(xs, r["mean"], w, label=n)
            if i == 0:
                for xx, v, s in zip(xs, r["mean"], r["size"]):
                    a.text(xx, v + 0.01, f"n={s}", ha="center", fontsize=6)
            a.set_xticks(np.arange(len(r)) + w * (len(errs) - 1) / 2, r.index.astype(str))
        a.set_title(f"recall vs {title}", fontsize=9)
        a.set_ylim(0, 1)
    ax[0].legend(fontsize=7)
    conf = gt[gt["confused_with"] >= 0]
    cm = pd.crosstab(conf["cls"], conf["confused_with"]).reindex(index=range(8), columns=range(8), fill_value=0)
    ax[3].imshow(cm.values, cmap="Reds")
    for (i, j), v in np.ndenumerate(cm.values):
        if v:
            ax[3].text(j, i, v, ha="center", va="center", fontsize=7)
    ax[3].set_xticks(range(8), common.CLASSES, rotation=45, fontsize=7)
    ax[3].set_yticks(range(8), common.CLASSES, fontsize=7)
    ax[3].set_xlabel("predicted as")
    ax[3].set_ylabel("GT class")
    ax[3].set_title(f"{ANALYSIS_EXPERIMENT}: missed GT confused with another class", fontsize=9)
    fig.savefig(IMG_DIR / "02b_retina_error_breakdown.png")
    plt.show()

**Спостереження (типи помилок):** TBD

### 9.3 Failure cases
Чотири кадри обрані автоматично: найбільше пропущених малих об'єктів (< 16 px), найбільше сплутувань класів,
найщільніший кадр, найбільше дублікатів (типова проблема без другого етапу). Кольори: зелений — TP, червоний —
пропущений GT, пурпуровий — неправильний клас, помаранчевий — погана локалізація, синій — дублікат, жовтий — FP на
фоні; сірий — ignored region. Кропи центровані на помилках.

In [ ]:
if DO_ANALYSIS:
    per = gt.assign(small_miss=(~gt["matched"]) & (gt["side"] < 16), conf=gt["confused_with"] >= 0)
    per = per.groupby("img_id").agg(small_miss=("small_miss", "sum"), conf=("conf", "sum"),
                                    n=("cls", "size"), recall=("matched", "mean"))
    per["dup"] = dt[dt["type"] == "dup"].groupby("img_id").size().reindex(per.index, fill_value=0) if len(dt) else 0
    cases = {"most missed small objects": per["small_miss"].idxmax(),
             "most class confusions": per["conf"].idxmax(),
             "densest frame": per["n"].idxmax(),
             "most duplicates": per["dup"].idxmax()}
    fig, ax = plt.subplots(2, 2, figsize=(16, 10))
    for a, (title, iid) in zip(ax.ravel(), cases.items()):
        a.imshow(tvt.draw_errors(iid, images, anns, gt, dt, crop="auto"))
        r = per.loc[iid]
        a.set_title(f"{title}: {images.loc[iid, 'clip']} #{images.loc[iid, 'frame_id']} "
                    f"(n={r['n']}, recall={r['recall']:.2f})", fontsize=9)
        a.axis("off")
    fig.savefig(IMG_DIR / "02b_retina_failure_cases.png")
    plt.show()

**Failure case 1 — дрібні об'єкти:** TBD

**Failure case 2 — сплутування класів:** TBD

**Failure case 3 — щільна сцена:** TBD

**Failure case 4 — дублікати:** TBD

## 10. Вихідні файли
У FULL-режимі зливаємо `results.csv` цієї сесії з копією з репозиторію в `results_merged.csv`
(завантажити в репозиторій як `notebooks/results.csv`) і показуємо список картинок для `images/`.

In [ ]:
if MODE == "full" and RESULTS.exists():
    merged = WORK / "results_merged.csv"
    try:
        urllib.request.urlretrieve(REPO_RAW + "results.csv", merged)
    except Exception as e:
        print("repo results.csv not available:", e)
        merged.unlink(missing_ok=True)
    common.merge_results(str(RESULTS), str(merged))
    print(f"{merged}: {len(pd.read_csv(merged))} rows")
print("images:", sorted(p.name for p in IMG_DIR.glob("02b_*.png")))

## 11. Висновки
TBD